# Granite 4.1 — Basic Usage (MLX)

## Imports

In [1]:
from pprint import pprint

import mlx_lm

print("mlx-lm:", mlx_lm.__version__)

mlx-lm: 0.31.3


## Load Model and Tokenizer

In [2]:
MODEL_ID = "mlx-community/granite-4.1-3b-mxfp8"

model, tokenizer = mlx_lm.load(MODEL_ID)

print(f"Architecture: {model.model_type}")
print(f"Parameters: {mlx_lm.utils.get_total_parameters(model):,}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Architecture: granite
Parameters: 3,402,836,480


## Single Turn Generation

In [3]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|start_of_role|>user<|end_of_role|>What is the capital of France?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>


In [4]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

The capital of France is Paris.


## System Prompt

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(chat)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant who responds in all capitals.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>What is the capital of France?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>


In [6]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

THE CAPITAL OF FRANCE IS PARIS.


## Multi-Turn Generation

In [7]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
response = mlx_lm.generate(model, tokenizer, inputs, max_tokens=128)

print(response)

The capital of France is Paris.


In [8]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'}]


In [9]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [10]:
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
response = mlx_lm.generate(model, tokenizer, inputs, max_tokens=128)

print(response)

The capital of France is Paris.


## Streaming Generation

In [11]:
messages = [
    {"role": "user", "content": "What is a capital of France?"},
]

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

for response in mlx_lm.stream_generate(model, tokenizer, inputs, max_tokens=128):
    print(response.text, end="", flush=True)

The capital of France is Paris.